In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [2]:
train_df = pd.read_parquet("../data/task_a/train.parquet")
val_df   = pd.read_parquet("../data/task_a/val.parquet")

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 500000
Validation size: 100000


#### TF-IDF Vectorizer

In [3]:
vectorizer = TfidfVectorizer(
    max_features=150000,      # limit features for 8GB RAM
    ngram_range=(1,1),        # unigram only
    analyzer="word",
    token_pattern=r'\b\w+\b', # keep code tokens
    lowercase=False           # case matters in code
)

#### Fit on Training Data

In [4]:
print("Fitting TF-IDF on training data...")

X_train = vectorizer.fit_transform(train_df['code'])
X_val   = vectorizer.transform(val_df['code'])

y_train = train_df['label']
y_val   = val_df['label']

print("TF-IDF train shape:", X_train.shape)
print("TF-IDF val shape:", X_val.shape)

Fitting TF-IDF on training data...
TF-IDF train shape: (500000, 150000)
TF-IDF val shape: (100000, 150000)


#### Train Logistic Regression

In [5]:
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

print("Training Logistic Regression...")
model.fit(X_train, y_train)

Training Logistic Regression...


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


#### Evaluate on Validation Set 

In [6]:
val_preds = model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Validation F1:", f1_score(y_val, val_preds))

print("\nClassification Report:")
print(classification_report(y_val, val_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_preds))

Validation Accuracy: 0.86873
Validation F1: 0.8658799489144317

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.93      0.87     47695
           1       0.93      0.81      0.87     52305

    accuracy                           0.87    100000
   macro avg       0.87      0.87      0.87    100000
weighted avg       0.88      0.87      0.87    100000


Confusion Matrix:
[[44499  3196]
 [ 9931 42374]]


#### Per-Language Validation Performance

In [7]:
val_df['pred'] = val_preds

for lang in val_df['language'].unique():
    subset = val_df[val_df['language'] == lang]
    acc = accuracy_score(subset['label'], subset['pred'])
    f1  = f1_score(subset['label'], subset['pred'])
    print(f"{lang} - Accuracy: {acc:.4f}, F1: {f1:.4f}")

Python - Accuracy: 0.8724, F1: 0.8692
C++ - Accuracy: 0.8451, F1: 0.8493
Java - Accuracy: 0.8104, F1: 0.8101


### Evaluate on Evaluation Set (OOD)

In [8]:
# Load evaluation set
eval_df = pd.read_parquet("../data/task_a/test_sample.parquet")

X_eval = vectorizer.transform(eval_df['code'])
y_eval = eval_df['label']

eval_preds = model.predict(X_eval)

from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Evaluation Accuracy:", accuracy_score(y_eval, eval_preds))
print("Evaluation F1:", f1_score(y_eval, eval_preds))

print("\nClassification Report:")
print(classification_report(y_eval, eval_preds))

Evaluation Accuracy: 0.39
Evaluation F1: 0.39723320158102765

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.24      0.38       777
           1       0.25      0.90      0.40       223

    accuracy                           0.39      1000
   macro avg       0.58      0.57      0.39      1000
weighted avg       0.75      0.39      0.39      1000



In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Predictions on test sample
eval_preds = model.predict(X_eval)
eval_probs = model.predict_proba(X_eval)[:, 1]

# Metrics
accuracy = accuracy_score(y_eval, eval_preds)
macro_precision = precision_score(y_eval, eval_preds, average="macro")
macro_recall = recall_score(y_eval, eval_preds, average="macro")
macro_f1 = f1_score(y_eval, eval_preds, average="macro")
binary_f1 = f1_score(y_eval, eval_preds)
roc_auc = roc_auc_score(y_eval, eval_probs)

print("====== TF-IDF + Logistic Regression (Baseline) - Test Sample ======")
print("Accuracy:", accuracy)
print("Macro Precision:", macro_precision)
print("Macro Recall:", macro_recall)
print("Macro F1:", macro_f1)
print("Binary F1:", binary_f1)
print("ROC AUC:", roc_auc)

print("\nClassification Report")
print(classification_report(y_eval, eval_preds))

====== TF-IDF + Logistic Regression (Baseline) - Test Sample ======
Accuracy: 0.39
Macro Precision: 0.5752437244337124
Macro Recall: 0.572294267361532
Macro F1: 0.3899121473492183
Binary F1: 0.39723320158102765
ROC AUC: 0.7053921314011

Classification Report
              precision    recall  f1-score   support

           0       0.90      0.24      0.38       777
           1       0.25      0.90      0.40       223

    accuracy                           0.39      1000
   macro avg       0.58      0.57      0.39      1000
weighted avg       0.75      0.39      0.39      1000



### Per-Language OOD Performance

In [10]:
eval_df['pred'] = eval_preds

for lang in eval_df['language'].unique():
    subset = eval_df[eval_df['language'] == lang]
    acc = accuracy_score(subset['label'], subset['pred'])
    f1  = f1_score(subset['label'], subset['pred'])
    print(f"{lang} - Accuracy: {acc:.4f}, F1: {f1:.4f}")

C# - Accuracy: 0.2951, F1: 0.3944
Go - Accuracy: 0.3667, F1: 0.4571
Python - Accuracy: 0.5017, F1: 0.4881
C++ - Accuracy: 0.4267, F1: 0.3768
Java - Accuracy: 0.3477, F1: 0.2894
PHP - Accuracy: 0.2292, F1: 0.1395
JavaScript - Accuracy: 0.3529, F1: 0.4954
C - Accuracy: 0.3529, F1: 0.3265


TF-IDF models achieve high in-domain performance but fail catastrophically under cross-language distribution shift.

## Improved TF-IDF Configuration

In [11]:
vectorizer = TfidfVectorizer(
    max_features=120000,      # reduced for safety
    ngram_range=(1,2),        # add bigrams
    analyzer="word",
    token_pattern=r'\b\w+\b',
    lowercase=False,
    min_df=5,                 # remove rare tokens
    sublinear_tf=True         # better generalization
)

#### Fit & Transform

In [12]:
print("Fitting improved TF-IDF...")

X_train = vectorizer.fit_transform(train_df['code'])
X_val   = vectorizer.transform(val_df['code'])
X_eval  = vectorizer.transform(eval_df['code'])

y_train = train_df['label']
y_val   = val_df['label']
y_eval  = eval_df['label']

print("Train shape:", X_train.shape)

Fitting improved TF-IDF...
Train shape: (500000, 120000)


#### Logistic Regression (Balanced)

In [13]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",  # handle label shift
    n_jobs=-1
)

print("Training model...")
model.fit(X_train, y_train)

Training model...


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


#### Evaluate on Validation

In [14]:
val_preds = model.predict(X_val)

print("=== Validation Performance ===")
print("Accuracy:", accuracy_score(y_val, val_preds))
print("F1:", f1_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

=== Validation Performance ===
Accuracy: 0.90176
F1: 0.901170979035049
              precision    recall  f1-score   support

           0       0.86      0.95      0.90     47695
           1       0.95      0.86      0.90     52305

    accuracy                           0.90    100000
   macro avg       0.90      0.90      0.90    100000
weighted avg       0.91      0.90      0.90    100000



#### Evaluate on Evaluation Set

In [15]:
eval_preds = model.predict(X_eval)

print("=== Evaluation Performance (OOD) ===")
print("Accuracy:", accuracy_score(y_eval, eval_preds))
print("F1:", f1_score(y_eval, eval_preds))
print(classification_report(y_eval, eval_preds))

=== Evaluation Performance (OOD) ===
Accuracy: 0.32
F1: 0.38405797101449274
              precision    recall  f1-score   support

           0       0.91      0.14      0.24       777
           1       0.24      0.95      0.38       223

    accuracy                           0.32      1000
   macro avg       0.57      0.54      0.31      1000
weighted avg       0.76      0.32      0.27      1000



#### Per-Language OOD Performance

In [16]:
eval_df['pred'] = eval_preds

print("\nPer-language Evaluation Performance:")
for lang in eval_df['language'].unique():
    subset = eval_df[eval_df['language'] == lang]
    acc = accuracy_score(subset['label'], subset['pred'])
    f1  = f1_score(subset['label'], subset['pred'])
    print(f"{lang}: Accuracy={acc:.4f}, F1={f1:.4f}")


Per-language Evaluation Performance:
C#: Accuracy=0.2459, F1=0.3784
Go: Accuracy=0.3500, F1=0.4507
Python: Accuracy=0.4653, F1=0.4808
C++: Accuracy=0.3200, F1=0.3377
Java: Accuracy=0.2070, F1=0.2827
PHP: Accuracy=0.1042, F1=0.1224
JavaScript: Accuracy=0.3647, F1=0.5091
C: Accuracy=0.2941, F1=0.3333


## Character-Level TF-IDF

- We use character n-grams of length 3–5.
- 1–2 → too small, too noisy
- 6+ → too sparse

In [17]:
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=100000,   # safe for 8GB
    min_df=5,
    sublinear_tf=True
)

#### Fit & Transform

In [18]:
print("Fitting Character TF-IDF...")

X_train = vectorizer.fit_transform(train_df['code'])
X_val   = vectorizer.transform(val_df['code'])
X_eval  = vectorizer.transform(eval_df['code'])

y_train = train_df['label']
y_val   = val_df['label']
y_eval  = eval_df['label']

print("Train shape:", X_train.shape)

Fitting Character TF-IDF...
Train shape: (500000, 100000)


#### Logistic Regression

In [19]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

print("Training model...")
model.fit(X_train, y_train)

Training model...


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


#### Validation Performance

In [20]:
val_preds = model.predict(X_val)

print("=== Validation Performance ===")
print("Accuracy:", accuracy_score(y_val, val_preds))
print("F1:", f1_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

=== Validation Performance ===
Accuracy: 0.95766
F1: 0.9590553922327093
              precision    recall  f1-score   support

           0       0.94      0.97      0.96     47695
           1       0.97      0.95      0.96     52305

    accuracy                           0.96    100000
   macro avg       0.96      0.96      0.96    100000
weighted avg       0.96      0.96      0.96    100000



#### Evaluation Performance

In [21]:
eval_preds = model.predict(X_eval)

print("=== Evaluation Performance (OOD) ===")
print("Accuracy:", accuracy_score(y_eval, eval_preds))
print("F1:", f1_score(y_eval, eval_preds))
print(classification_report(y_eval, eval_preds))

=== Evaluation Performance (OOD) ===
Accuracy: 0.296
F1: 0.38461538461538464
              precision    recall  f1-score   support

           0       0.96      0.10      0.18       777
           1       0.24      0.99      0.38       223

    accuracy                           0.30      1000
   macro avg       0.60      0.54      0.28      1000
weighted avg       0.80      0.30      0.22      1000



#### Per-Language OOD

In [22]:
eval_df['pred'] = eval_preds

print("\nPer-language Evaluation Performance:")
for lang in eval_df['language'].unique():
    subset = eval_df[eval_df['language'] == lang]
    acc = accuracy_score(subset['label'], subset['pred'])
    f1  = f1_score(subset['label'], subset['pred'])
    print(f"{lang}: Accuracy={acc:.4f}, F1={f1:.4f}")


Per-language Evaluation Performance:
C#: Accuracy=0.2295, F1=0.3733
Go: Accuracy=0.2667, F1=0.4211
Python: Accuracy=0.4488, F1=0.4830
C++: Accuracy=0.2933, F1=0.3291
Java: Accuracy=0.1914, F1=0.2837
PHP: Accuracy=0.0625, F1=0.1176
JavaScript: Accuracy=0.3765, F1=0.5470
C: Accuracy=0.1961, F1=0.3051


## Balanced-Language Training Experiment

### Create Balanced Training Set

In [23]:
# Create language-balanced subset

balanced_samples = []

languages = ['Python', 'C++', 'Java']
samples_per_language = 15000

for lang in languages:
    subset = train_df[train_df['language'] == lang]
    subset_sampled = subset.sample(n=samples_per_language, random_state=42)
    balanced_samples.append(subset_sampled)

balanced_train_df = pd.concat(balanced_samples)

print("Balanced training size:", len(balanced_train_df))
print(balanced_train_df['language'].value_counts())
print(balanced_train_df['label'].value_counts(normalize=True))

Balanced training size: 45000
language
Python    15000
C++       15000
Java      15000
Name: count, dtype: int64
label
1    0.5224
0    0.4776
Name: proportion, dtype: float64


#### Character TF-IDF on Balanced Data

In [24]:
vectorizer_balanced = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=100000,
    min_df=5,
    sublinear_tf=True
)

print("Fitting balanced Character TF-IDF...")

X_train_bal = vectorizer_balanced.fit_transform(balanced_train_df['code'])
X_val_bal   = vectorizer_balanced.transform(val_df['code'])
X_eval_bal  = vectorizer_balanced.transform(eval_df['code'])

y_train_bal = balanced_train_df['label']
y_val = val_df['label']
y_eval = eval_df['label']

Fitting balanced Character TF-IDF...

#### Train Model

In [25]:
model_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

print("Training balanced model...")
model_balanced.fit(X_train_bal, y_train_bal)

Training balanced model...


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


### Evaluate

#### Validation

In [26]:
val_preds_bal = model_balanced.predict(X_val_bal)

print("=== Validation Performance (Balanced Training) ===")
print("Accuracy:", accuracy_score(y_val, val_preds_bal))
print("F1:", f1_score(y_val, val_preds_bal))

=== Validation Performance (Balanced Training) ===
Accuracy: 0.92901
F1: 0.930929470027924


#### Evaluation test sample

In [27]:
eval_preds_bal = model_balanced.predict(X_eval_bal)

print("=== Evaluation Performance (Balanced Training) ===")
print("Accuracy:", accuracy_score(y_eval, eval_preds_bal))
print("F1:", f1_score(y_eval, eval_preds_bal))

=== Evaluation Performance (Balanced Training) ===
Accuracy: 0.285
F1: 0.38095238095238093


Hence we can say Balanced training did not improve OOD performance.
